# moscot synthetic dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit moscot and export forecasts

In [ ]:
# Synthetic Developmental Crescent v1. EXACT frozen public data; no oracle files.
# Reproduces the *spatiotemporal tutorial* choice of low-rank FGW parameters.
# MODE='tutorial': alpha=.5, epsilon=0, rank=200 (not identical to API defaults).
# MODE='api_defaults': alpha=.5, epsilon=.001, rank=-1 (optional, not automatic).
# Does not run a hyperparameter sweep or change settings based on held-out truth.

from pathlib import Path
import os, sys, hashlib, json, time, platform, inspect, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path(os.environ.get(
    'STP_PROJECT', os.environ["STP_PROJECT"]))
DATA_ROOT = Path(os.environ.get('CRESCENT_DATA_ROOT', str(
    PROJECT / 'Experiments/simulation validation/Synthetic benchmarking/experiments/synthetic_developmental_crescent_v1')))
MODE = os.environ.get('MOSCOT_MODE', 'tutorial')  # 'tutorial' or 'api_defaults'
assert MODE in {'tutorial', 'api_defaults'}
PRE_FLIGHT_ONLY = os.environ.get('MOSCOT_PREFLIGHT_ONLY', '0') == '1'
OUT_ROOT = Path(os.environ.get('MOSCOT_CRESCENT_OUT', str(
    PROJECT / 'experiments/synthetic_benchmark/synthetic_developmental_crescent_v1_moscot'))) / MODE
SOURCE_STAGE, TARGET_STAGE, LAMBDA_INTERP = 2, 3, 0.5
SOURCE_T, PRED_T = 2/9, 2.5/9
GENES = [f'G{i:02d}' for i in range(1,49)]


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(data, sort_keys=True, indent=2, default=str)+'\n')
    os.replace(temporary, path)


# Dataset integrity. NEVER read DATA_ROOT/evaluator_only.
manifest_path = DATA_ROOT/'manifests/dataset_manifest.json'
assert manifest_path.is_file(), f'Frozen generator manifest missing: {manifest_path}'
manifest = json.loads(manifest_path.read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert manifest['seed'] == 20261023 and manifest['status'] == 'frozen'
assert manifest['n_observed_cells'] == 36425 and manifest['n_source_probes'] == 363
assert manifest['genes'] == GENES
assert np.isclose(manifest['source_t'], SOURCE_T)
assert np.isclose(manifest['target_t'], PRED_T)
input_rel = (
    'train_only/public/observed_cells.csv.gz',
    'train_only/public/counts.csv.gz',
    'train_only/public/genes.csv',
    'manifests/source_probe_cell_ids.csv.gz',
)
input_sha = {}
for rel in input_rel:
    path = DATA_ROOT / rel
    assert path.is_file(), path
    checksum = sha256(path)
    assert manifest['file_sha256'][rel] == checksum, f'Frozen file changed: {rel}'
    input_sha[rel] = checksum

public = DATA_ROOT/'train_only/public'
obs = pd.read_csv(public/'observed_cells.csv.gz')
count_df = pd.read_csv(public/'counts.csv.gz')
gene_file = pd.read_csv(public/'genes.csv')
probe_ids = pd.read_csv(DATA_ROOT/'manifests/source_probe_cell_ids.csv.gz')['cell_id'].astype(str).to_numpy()
assert list(obs.columns) == ['cell_id','tissue_id','time','x','y']
assert list(count_df.columns) == ['cell_id', *GENES]
assert gene_file['gene'].tolist() == GENES
assert len(obs) == len(count_df) == 36425
assert obs['cell_id'].is_unique and count_df['cell_id'].is_unique
assert np.array_equal(obs.cell_id.to_numpy(str), count_df.cell_id.to_numpy(str))
assert np.array_equal(np.sort(obs.tissue_id.unique()), np.arange(10))
assert np.allclose(np.sort(obs.time.unique()), np.linspace(0,1,10))
assert not np.isclose(obs.time.to_numpy(), PRED_T).any()
assert np.allclose(obs.time, obs.tissue_id / 9)
assert len(probe_ids) == len(set(probe_ids)) == 363
src_check = obs.loc[obs.tissue_id.eq(SOURCE_STAGE),'cell_id'].to_numpy(str)
assert set(probe_ids).issubset(set(src_check))
X = count_df[GENES].to_numpy(dtype=np.float32,copy=True)
XY = obs[['x','y']].to_numpy(dtype=np.float32,copy=True)
assert np.isfinite(X).all() and (X >= 0).all() and np.array_equal(X,np.rint(X))
assert np.isfinite(XY).all()
assert len(src_check) == 2419
print('LOCKED CRESCENT v1:', DATA_ROOT)
print('Observed cells:', len(obs), '| genes:', len(GENES), '| source probe IDs:',len(probe_ids))
print('Observed tissue counts:', obs.groupby('tissue_id').size().to_dict())
print('Held-out time:', PRED_T, '| available only to evaluator after fitting')
print('Frozen public-file SHA-256 hashes verified; no hidden files accessed.')

if not PRE_FLIGHT_ONLY:
    import anndata as ad
    import scanpy as sc
    import moscot
    import jax
    from moscot.problems.spatiotemporal import SpatioTemporalProblem

    print('moscot', importlib.metadata.version('moscot'),
          '| scanpy',sc.__version__, '| jax',jax.__version__)
    print('JAX devices:',jax.devices())
    # Run on the JAX GPU kernel / appropriate CUDA build. Fail rather than
    # silently run the computationally intensive main solver on CPU.
    assert any(d.platform == 'gpu' for d in jax.devices()), (
        'No JAX GPU available. Install the JAX/OTT GPU versions compatible with your CUDA stack.')

    if OUT_ROOT.exists() and (OUT_ROOT/'complete.json').exists():
        complete = json.loads((OUT_ROOT/'complete.json').read_text())
        assert complete['data_sha256'] == input_sha, 'Completed run used different data'
        assert complete['mode'] == MODE
        for name, checksum in complete.get('file_sha256',{}).items():
            assert sha256(OUT_ROOT/name) == checksum, f'Completed artifact altered: {name}'
        print('ALREADY COMPLETE and VERIFIED; refusing to overwrite:',OUT_ROOT)
    else:
        if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
            raise RuntimeError(f'Existing incomplete directory {OUT_ROOT}; inspect/rename before running again.')
        OUT_ROOT.mkdir(parents=True,exist_ok=True)

        # Match the tutorial's preprocessed expression input (log-normalized counts).
        # All 48 genes are kept: synthetic genes do not carry biological marker labels.
        # No HVG/gene panel selection and no oracle coordinate adjustments.
        adata = ad.AnnData(X=X.copy())
        adata.obs_names = pd.Index(obs.cell_id.astype(str).to_numpy(),name='cell_id')
        adata.var_names = pd.Index(GENES,name='gene')
        adata.obs['time_bin'] = pd.Categorical(
            obs.tissue_id.to_numpy(int),categories=list(range(10)),ordered=True)
        adata.obs['time_fraction'] = obs.time.to_numpy(float)
        adata.obsm['spatial'] = XY.copy()
        adata.layers['counts'] = X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        assert np.isfinite(adata.X).all()

        config = {
            'dataset_id':manifest['dataset_id'], 'dataset_seed':manifest['seed'],
            'input_sha256':input_sha, 'mode':MODE,
            'moscot_version':importlib.metadata.version('moscot'),
            'scanpy_version':sc.__version__, 'jax_version':jax.__version__,
            'python':sys.version, 'platform':platform.platform(),
            'time_key':'time_bin', 'spatial_key':'spatial',
            'source_time_index':SOURCE_STAGE, 'target_time_index':TARGET_STAGE,
            'unobserved_time':PRED_T,
            'expression_preprocessing':'scanpy normalize_total target_sum=1e4 then log1p; retain all 48 genes',
            'spatial_preprocessing':'raw observed sequencer x/y in obsm[spatial]; normalize_spatial=True default',
            'marginals':'uniform (no marker-gene scoring: synthetic panel has no proliferation/apoptosis marker genes)',
            'joint_cost':'tutorial local PCA on log-normalized data; 30 PCs',
            'pair_policy':'sequential across all 10 observed snapshots',
            'solver': {'alpha':0.5,'epsilon':0.0,'rank':200} if MODE == 'tutorial' else
                      {'alpha':0.5,'epsilon':0.001,'rank':-1},
            'metadata':'Tutorial low-rank settings are explicitly specified, not the moscot solve() defaults.',
        }
        write_json(OUT_ROOT/'run_config.json',config)
        adata.write_h5ad(OUT_ROOT/'public_input_logCP10k.h5ad')

        # Current API uses xy_callback; the tutorial printed 'callback' in some
        # documentation releases. We inspect the installed signature and record it.
        sig = inspect.signature(SpatioTemporalProblem.prepare)
        arg = 'xy_callback' if 'xy_callback' in sig.parameters else 'callback'
        if arg not in sig.parameters:
            raise RuntimeError('Installed SpatioTemporalProblem.prepare lacks local-PCA callback API')
        config['prepare_callback_argument'] = arg
        write_json(OUT_ROOT/'run_config.json',config)
        stp = SpatioTemporalProblem(adata)
        stp = stp.prepare(
            time_key='time_bin', spatial_key='spatial', joint_attr=None,
            normalize_spatial=True, policy='sequential', **{arg:'local-pca'})
        actual_pairs = [tuple(map(int,p)) for p in stp.problems]
        expected_pairs = [(i,i+1) for i in range(9)]
        assert sorted(actual_pairs) == expected_pairs, (actual_pairs,expected_pairs)
        print('Prepared 9 tutorial sequential FGW subproblems:',actual_pairs)

        start = time.time()
        stp = stp.solve(**config['solver'])
        duration = time.time()-start
        print('moscot solve completed in',round(duration,1),'seconds')

        import jax.numpy as jnp
        stats = []
        out_hashes = {}
        pair_root = OUT_ROOT/'pairs'
        pair_root.mkdir(exist_ok=True)
        # Materialize each native fitted coupling; full matrices are modest for
        # these ~2k-6k-cell tissue pairs. All arrays mapped by exact IDs.
        for source, target in expected_pairs:
            sol = stp[(source,target)].solution
            assert sol is not None, (source,target)
            P = np.asarray(sol.transport_matrix, dtype=np.float32)
            a_ids = obs.loc[obs.tissue_id.eq(source),'cell_id'].to_numpy(str)
            b_ids = obs.loc[obs.tissue_id.eq(target),'cell_id'].to_numpy(str)
            assert P.shape == (len(a_ids),len(b_ids)), (source,target,P.shape)
            assert np.isfinite(P).all() and np.min(P) >= -1e-7
            P = np.maximum(P,0)
            mass = float(P.sum())
            assert mass > 0
            row_mass=P.sum(axis=1)
            col_mass=P.sum(axis=0)
            converged=getattr(sol, 'converged', None)
            # Preserve convergence info, DO NOT tune to force convergence.
            item = dict(pair=[source,target],shape=list(P.shape),total_mass=mass,
                        zero_rows=int((row_mass<=1e-12).sum()),
                        row_mass_cv=float(row_mass.std()/max(row_mass.mean(),1e-12)),
                        column_mass_cv=float(col_mass.std()/max(col_mass.mean(),1e-12)),
                        converged=None if converged is None else bool(np.asarray(converged)),
                        output_rank=int(np.linalg.matrix_rank(P[:min(150,len(P)),:min(150,P.shape[1])])) if False else None)
            stats.append(item)
            pdirectory = pair_root/f't{source:02d}_to_t{target:02d}'
            pdirectory.mkdir(exist_ok=True)
            for filename, array in [('transport.npy',P),('source_ids.npy',a_ids),('target_ids.npy',b_ids)]:
                dest=pdirectory/filename
                np.save(dest,array,allow_pickle=False)
                out_hashes[dest.relative_to(OUT_ROOT).as_posix()] = sha256(dest)
            print(f'Pair {source}->{target}: {P.shape}, mass={mass:.6f}, '
                  f'zero rows={item["zero_rows"]}, converged={item["converged"]}', flush=True)
            del P

        # Source-conditioned point prediction at t=2.5/9 for comparable
        # cell-matched scoring. This is our disclosed evaluation adapter, NOT a
        # native deterministic lineage forecast from moscot.
        P = np.load(pair_root/'t02_to_t03/transport.npy', mmap_mode='r')
        source_ids = np.load(pair_root/'t02_to_t03/source_ids.npy',allow_pickle=False)
        target_ids = np.load(pair_root/'t02_to_t03/target_ids.npy',allow_pickle=False)
        assert np.array_equal(source_ids,src_check)
        cell_index = pd.Index(obs.cell_id.to_numpy(str))
        ia=cell_index.get_indexer(source_ids); ib=cell_index.get_indexer(target_ids)
        assert (ia >= 0).all() and (ib >= 0).all()
        source_expression = np.asarray(adata.X[ia],dtype=np.float64)
        target_expression = np.asarray(adata.X[ib],dtype=np.float64)
        weighted=np.asarray(P,dtype=np.float64) @ target_expression
        mass=np.asarray(P.sum(axis=1),dtype=np.float64)
        assert (mass > 1e-12).all(), 'Transport contains zero-mass source rows; preserve as QC failure'
        target_conditional=weighted/mass[:,None]
        interpolated_log = (.5*source_expression + .5*target_conditional).astype('float32')
        assert np.isfinite(interpolated_log).all()
        # Convert only for fair downstream scoring. The common evaluator will
        # be responsible for deciding the final metric / representation.
        rate_proxy=np.expm1(np.maximum(interpolated_log,0)).astype('float32')
        shares=rate_proxy/np.maximum(rate_proxy.sum(axis=1,keepdims=True),1e-12)
        forecast_file=OUT_ROOT/'moscot_t2_to_t2p5_frozen_source_forecast.npz'
        np.savez_compressed(forecast_file,
            source_ids=source_ids, genes=np.array(GENES,dtype=str),
            source_probe_ids=np.array(probe_ids,dtype=str),
            source_probe_mask=np.isin(source_ids,probe_ids),
            prediction_logCP10k=interpolated_log,
            pred_gene_share=shares.astype('float32'),
            t_source=np.array(SOURCE_T),t_pred=np.array(PRED_T),
            pair_source=SOURCE_STAGE,pair_target=TARGET_STAGE,
            interpolation_weight=LAMBDA_INTERP)
        out_hashes[forecast_file.name]=sha256(forecast_file)

        # Save native moscot object as its own model artifact for reproducibility.
        # If cloudpickle of a particular upstream version fails, preserve all
        # coupling matrices and fail visibly rather than claiming a saved model.
        native_path=OUT_ROOT/'native_spatiotemporal_problem.pkl'
        stp.save(native_path, overwrite=False)
        assert native_path.is_file()
        out_hashes[native_path.name]=sha256(native_path)

        # Plot purely from public data and native solution. high-z up *display only*.
        fig,axs=plt.subplots(1,3,figsize=(15,4),constrained_layout=True)
        axs[0].scatter(XY[ia,0],XY[ia,1],s=.4,alpha=.45,label='source 2/9')
        axs[0].scatter(XY[ib,0],XY[ib,1],s=.4,alpha=.35,label='target 3/9')
        axs[0].set_title('Observed source/target (no oracle)');axs[0].legend(markerscale=7)
        axs[1].hist(mass,bins=45);axs[1].set_title('Source OT marginal masses')
        gene=5
        axs[2].scatter(XY[ia,0],XY[ia,1],c=interpolated_log[:,gene],s=2,cmap='viridis')
        axs[2].set_title(f'Source-conditioned interpolated {GENES[gene]}')
        for ax in (axs[0],axs[2]):
            ax.invert_yaxis(); ax.set_aspect('equal',adjustable='box')
        pfig=OUT_ROOT/'public_only_solution_QC.png'
        fig.savefig(pfig,dpi=170);plt.show();plt.close(fig)
        out_hashes[pfig.name]=sha256(pfig)
        stat_file=OUT_ROOT/'pairwise_solution_qc.json'
        write_json(stat_file,stats)
        out_hashes[stat_file.name]=sha256(stat_file)
        out_hashes['run_config.json']=sha256(OUT_ROOT/'run_config.json')
        out_hashes['public_input_logCP10k.h5ad']=sha256(OUT_ROOT/'public_input_logCP10k.h5ad')
        write_json(OUT_ROOT/'complete.json',{
            'data_sha256':input_sha, 'mode':MODE,
            'fit_duration_seconds':duration, 'n_observed':len(obs),
            'n_genes':len(GENES), 'n_source':len(src_check),
            'n_source_probes':len(probe_ids),'target_time':PRED_T,
            'file_sha256':out_hashes,
            'all_solvers_converged':all(item['converged'] is True for item in stats),
        })
        print('COMPLETE:',OUT_ROOT)
        print('NOTE: source-conditioned forecast is an evaluation adapter, not moscot lineage prediction.')
else:
    print('PREFLIGHT PASSED: correct locked public data; model not imported or trained.')